# DeepSeek vs Manual Paragraph Comparisons


This notebook contrasts the DeepSeek-generated paragraph CSVs with the manually parsed versions for the 2014 Unilever and Nestlé annual reports. It profiles each dataset, evaluates alignment using TF-IDF cosine similarity, SequenceMatcher ratios, and token-level Jaccard scores, visualizes similarity distributions, and adds a filtered secondary analysis that excludes bullet-like, table-like, and very short (\<2 sentence) paragraphs.


In [ ]:

import pandas as pd
import numpy as np
import re
from pathlib import Path
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ImportError as exc:
    raise ImportError("scikit-learn is required for the TF-IDF similarity calculations.") from exc

pd.set_option("display.max_colwidth", 160)
plt.style.use('seaborn-v0_8-deep')
BASE_DIR = Path('.').resolve()

def read_csv_with_fallback(path, encodings=("utf-8", "utf-8-sig", "cp1252", "latin-1")):
    last_error = None
    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise UnicodeDecodeError(
        last_error.encoding if last_error else 'utf-8',
        last_error.object if last_error else b'',
        last_error.start if last_error else 0,
        last_error.end if last_error else 0,
        f"Unable to decode {path} with encodings {encodings}. Last error: {last_error}"
    )


In [ ]:

auto_files = {
    "unilever": BASE_DIR / "unilever_ar14_paras.csv",
    "nestle": BASE_DIR / "nestle_2014_paras.csv",
}

manual_files = {
    "unilever": BASE_DIR / "unilever-2014-manual-parsing.csv",
    "nestle": BASE_DIR / "nestle-2014-manual-parsing.csv",
}

for label, path in {**auto_files, **manual_files}.items():
    if not path.exists():
        raise FileNotFoundError(f"Expected file missing: {path}")

auto_dfs = {name: read_csv_with_fallback(path) for name, path in auto_files.items()}
manual_dfs = {name: read_csv_with_fallback(path) for name, path in manual_files.items()}

auto_dfs["unilever"].head()


In [ ]:

def dataset_profile(name, df, text_col):
    text = df[text_col].fillna('').astype(str)
    return {
        "dataset": name,
        "rows": len(df),
        "empty_text_rows": int((text.str.strip() == '').sum()),
        "avg_char_len": float(text.str.len().mean()),
        "median_char_len": float(text.str.len().median()),
        "avg_word_len": float(text.str.split().str.len().mean()),
    }

profiles_full = pd.DataFrame([
    dataset_profile("unilever_deepseek", auto_dfs["unilever"], "text"),
    dataset_profile("unilever_manual", manual_dfs["unilever"], "paragraph"),
    dataset_profile("nestle_deepseek", auto_dfs["nestle"], "text"),
    dataset_profile("nestle_manual", manual_dfs["nestle"], "paragraph"),
]).assign(scenario="Full")

profiles_full


## Text & Similarity Utilities

In [ ]:

bullet_prefixes = ('-', '*', '•', '▪', '●', '‣', '◦', '–', '—')

DOC_CONFIGS = {
    "unilever": {
        "label": "Unilever 2014",
        "manual_id_col": "company",
        "manual_text_col": "paragraph",
        "auto_text_col": "text",
        "auto_id_cols": ["page", "para_index"],
    },
    "nestle": {
        "label": "Nestlé 2014",
        "manual_id_col": "company",
        "manual_text_col": "paragraph",
        "auto_text_col": "text",
        "auto_id_cols": ["page", "para_index"],
    },
}

def sentence_count(text: str) -> int:
    if not isinstance(text, str):
        return 0
    stripped = text.strip()
    if not stripped:
        return 0
    sentences = [s for s in re.split(r'(?<=[.!?])\s+', stripped) if s.strip()]
    return len(sentences)

def is_bullet_like(text: str) -> bool:
    if not isinstance(text, str):
        return False
    stripped = text.strip()
    if not stripped:
        return True
    if stripped.startswith(bullet_prefixes):
        return True
    if re.match(r"^\d+[\).]\s", stripped):
        return True
    lines = [line.strip() for line in stripped.splitlines() if line.strip()]
    if lines and all(line.startswith(bullet_prefixes) for line in lines):
        return True
    return False

def is_table_like(text: str) -> bool:
    if not isinstance(text, str):
        return False
    if '|' in text or '	' in text:
        return True
    lowered = text.lower()
    if lowered.startswith('table ') or lowered.startswith('table:'):
        return True
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) >= 3:
        punctuated = sum(1 for line in lines if re.search(r'[.!?]$', line))
        numeric_lines = sum(1 for line in lines if sum(ch.isdigit() for ch in line) >= max(1, int(0.2 * len(line.replace(' ', '')))))
        if punctuated / len(lines) < 0.2 and numeric_lines >= len(lines) * 0.5:
            return True
    return False

def should_keep_paragraph(text: str) -> bool:
    if not isinstance(text, str):
        return False
    if is_bullet_like(text) or is_table_like(text):
        return False
    return True


def is_short_or_fragment(text: str, min_chars: int = 80, min_sentences: int = 2) -> bool:
    text = (text or '').strip()
    if not text:
        return True
    if len(text) < min_chars:
        return True
    if sentence_count(text) < min_sentences:
        return True
    return False

def filter_paragraph_df(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    mask = df[text_col].fillna('').map(should_keep_paragraph)
    return df[mask].reset_index(drop=True)


def filter_paragraph_df_without_fragments(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    mask = df[text_col].fillna('').map(lambda t: should_keep_paragraph(t) and not is_short_or_fragment(t))
    return df[mask].reset_index(drop=True)

def normalize_text(value: str) -> str:
    if not isinstance(value, str):
        return ''
    return re.sub(r"\s+", " ", value.strip()).lower()

def token_set(value: str) -> set:
    return set(normalize_text(value).split())

def jaccard_similarity(a: str, b: str) -> float:
    tokens_a = token_set(a)
    tokens_b = token_set(b)
    if not tokens_a and not tokens_b:
        return 1.0
    union = tokens_a | tokens_b
    if not union:
        return 0.0
    return len(tokens_a & tokens_b) / len(union)

def sequence_ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

def build_similarity_frame(
    manual_df: pd.DataFrame,
    manual_id_col: str,
    manual_text_col: str,
    auto_df: pd.DataFrame,
    auto_text_col: str,
    auto_id_cols,
    doc_label: str,
    top_k: int = 3,
) -> pd.DataFrame:
    if manual_df.empty or auto_df.empty:
        return pd.DataFrame()
    manual_df = manual_df.copy()
    auto_df = auto_df.copy()
    manual_df["manual_text"] = manual_df[manual_text_col].fillna('').astype(str)
    auto_df["auto_text"] = auto_df[auto_text_col].fillna('').astype(str)

    manual_texts = manual_df["manual_text"].tolist()
    auto_texts = auto_df["auto_text"].tolist()
    vocab_source = manual_texts + auto_texts

    vectorizer = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
    vectorizer.fit(vocab_source)
    manual_matrix = vectorizer.transform(manual_texts)
    auto_matrix = vectorizer.transform(auto_texts)
    cosine_scores = cosine_similarity(manual_matrix, auto_matrix)

    results = []
    for manual_idx, manual_row in manual_df.reset_index(drop=True).iterrows():
        ranked_indices = np.argsort(cosine_scores[manual_idx])[::-1][:top_k]
        manual_text = manual_row["manual_text"]
        manual_norm = normalize_text(manual_text)
        for rank, auto_idx in enumerate(ranked_indices, start=1):
            auto_row = auto_df.iloc[auto_idx]
            auto_text = auto_row["auto_text"]
            auto_norm = normalize_text(auto_text)
            auto_id = " | ".join(f"{col}={auto_row[col]}" for col in auto_id_cols)

            results.append({
                "document": doc_label,
                "manual_id": manual_row[manual_id_col],
                "auto_id": auto_id,
                "rank": rank,
                "cosine_similarity": float(cosine_scores[manual_idx, auto_idx]),
                "sequence_ratio": sequence_ratio(manual_norm, auto_norm),
                "jaccard_similarity": jaccard_similarity(manual_norm, auto_norm),
                "manual_char_len": len(manual_text),
                "auto_char_len": len(auto_text),
                "manual_preview": manual_text[:160],
                "auto_preview": auto_text[:160],
            })
    return pd.DataFrame(results)

def compute_analysis(auto_map, manual_map, configs, top_k: int = 3):
    similarity_frames = {}
    for key, cfg in configs.items():
        manual_df = manual_map.get(key)
        auto_df = auto_map.get(key)
        if manual_df is None or auto_df is None or manual_df.empty or auto_df.empty:
            continue
        similarity_frames[cfg['label']] = build_similarity_frame(
            manual_df=manual_df,
            manual_id_col=cfg['manual_id_col'],
            manual_text_col=cfg['manual_text_col'],
            auto_df=auto_df,
            auto_text_col=cfg['auto_text_col'],
            auto_id_cols=cfg['auto_id_cols'],
            doc_label=cfg['label'],
            top_k=top_k,
        )

    if not similarity_frames:
        return {
            'similarity_frames': {},
            'combined': pd.DataFrame(),
            'rank1': pd.DataFrame(),
            'auto_best': pd.DataFrame(),
            'coverage': pd.DataFrame(),
        }

    combined = pd.concat(similarity_frames.values(), ignore_index=True)
    rank1 = combined.query("rank == 1").copy()
    if not rank1.empty:
        rank1.loc[:, 'length_ratio'] = rank1['auto_char_len'] / rank1['manual_char_len'].replace(0, np.nan)
        rank1.loc[:, 'length_diff'] = rank1['auto_char_len'] - rank1['manual_char_len']
        rank1.loc[:, 'auto_page'] = rank1['auto_id'].str.extract(r'page=(\d+)').astype(float)
        rank1.loc[:, 'auto_para_index'] = rank1['auto_id'].str.extract(r'para_index=(\d+)').astype(float)

    auto_best = pd.DataFrame()
    if not rank1.empty:
        auto_best = (
            rank1.sort_values('cosine_similarity', ascending=False)
            .drop_duplicates(subset=['document', 'auto_id'])
        )

    coverage_records = []
    for key, cfg in configs.items():
        auto_df = auto_map.get(key)
        total = len(auto_df) if auto_df is not None else 0
        matched = 0
        if total and not auto_best.empty:
            matched = auto_best[auto_best['document'] == cfg['label']]['auto_id'].nunique()
        coverage_records.append({
            'document': cfg['label'],
            'total_auto_paragraphs': total,
            'matched_auto_paragraphs': matched,
            'coverage_pct': (matched / total * 100) if total else 0.0,
        })

    coverage = pd.DataFrame(coverage_records)

    return {
        'similarity_frames': similarity_frames,
        'combined': combined,
        'rank1': rank1,
        'auto_best': auto_best,
        'coverage': coverage,
    }


## Filtered Copies (≥2 sentences, no bullets/tables)

In [ ]:
filtered_auto_dfs = {name: filter_paragraph_df(df, "text") for name, df in auto_dfs.items()}
filtered_manual_dfs = {name: filter_paragraph_df(df, "paragraph") for name, df in manual_dfs.items()}

short_filtered_auto_dfs = {name: filter_paragraph_df_without_fragments(df, "text") for name, df in auto_dfs.items()}
short_filtered_manual_dfs = filtered_manual_dfs

profiles_filtered = pd.DataFrame([
    dataset_profile("unilever_deepseek_filtered", filtered_auto_dfs["unilever"], "text"),
    dataset_profile("unilever_manual_filtered", filtered_manual_dfs["unilever"], "paragraph"),
    dataset_profile("nestle_deepseek_filtered", filtered_auto_dfs["nestle"], "text"),
    dataset_profile("nestle_manual_filtered", filtered_manual_dfs["nestle"], "paragraph"),
]).assign(scenario="Filtered")

profiles_short_filtered = pd.DataFrame([
    dataset_profile("unilever_deepseek_short_filtered", short_filtered_auto_dfs["unilever"], "text"),
    dataset_profile("unilever_manual_short_filtered", short_filtered_manual_dfs["unilever"], "paragraph"),
    dataset_profile("nestle_deepseek_short_filtered", short_filtered_auto_dfs["nestle"], "text"),
    dataset_profile("nestle_manual_short_filtered", short_filtered_manual_dfs["nestle"], "paragraph"),
]).assign(scenario="FilteredNoFragments")

profile_summary = pd.concat([profiles_full, profiles_filtered, profiles_short_filtered], ignore_index=True)
profile_summary


## Build Similarity Tables

In [ ]:
analysis_inputs = {
    "Full": {"auto": auto_dfs, "manual": manual_dfs},
    "Filtered": {"auto": filtered_auto_dfs, "manual": filtered_manual_dfs},
    "FilteredNoFragments": {"auto": short_filtered_auto_dfs, "manual": short_filtered_manual_dfs},
}

analysis_results = {}
for scenario, inputs in analysis_inputs.items():
    analysis_results[scenario] = compute_analysis(inputs["auto"], inputs["manual"], DOC_CONFIGS, top_k=3)

{scenario: list(result['similarity_frames'].keys()) for scenario, result in analysis_results.items()}


## Aggregate Similarity Statistics

In [ ]:

summary_tables = []
for scenario, result in analysis_results.items():
    rank1 = result['rank1']
    if rank1.empty:
        continue
    summary = (
        rank1.groupby('document')[['cosine_similarity', 'sequence_ratio', 'jaccard_similarity']]
        .agg(['mean', 'median', 'min', 'max'])
    )
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary = summary.reset_index()
    summary['analysis'] = scenario
    summary_tables.append(summary)

if summary_tables:
    similarity_summary = pd.concat(summary_tables, ignore_index=True)
else:
    similarity_summary = pd.DataFrame()

similarity_summary


## Similarity Distributions

In [ ]:

metrics = ['cosine_similarity', 'sequence_ratio', 'jaccard_similarity']
for scenario, result in analysis_results.items():
    rank1 = result['rank1']
    if rank1.empty:
        print(f"{scenario}: No rank-1 matches to plot")
        continue
    fig, axes = plt.subplots(len(metrics), 1, figsize=(10, 4 * len(metrics)), sharex=True)
    if len(metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, metrics):
        for doc, group in rank1.groupby('document'):
            ax.hist(group[metric], bins=20, alpha=0.6, label=doc)
        ax.set_title(f"{metric.replace('_', ' ').title()} Distribution ({scenario})")
        ax.set_xlabel('Score')
        ax.set_ylabel('Count')
        ax.set_xlim(0, 1)
        ax.legend()
    plt.tight_layout()
    plt.show()


## Extraction Length Diagnostics

In [ ]:

length_tables = []
for scenario, result in analysis_results.items():
    rank1 = result['rank1']
    if rank1.empty:
        continue
    summary = (
        rank1.groupby('document')[['length_ratio', 'length_diff']]
        .agg(['mean', 'median', 'min', 'max'])
    )
    summary.columns = [f"{metric}_{stat}" for metric, stat in summary.columns]
    summary = summary.reset_index()
    summary['analysis'] = scenario
    length_tables.append(summary)

length_summary = pd.concat(length_tables, ignore_index=True) if length_tables else pd.DataFrame()
length_summary


In [ ]:

for scenario, result in analysis_results.items():
    rank1 = result['rank1']
    if rank1.empty:
        print(f"{scenario}: No rank-1 matches to visualize")
        continue
    docs = rank1['document'].unique()
    fig, axes = plt.subplots(1, len(docs), figsize=(12, 5), sharey=True)
    if len(docs) == 1:
        axes = [axes]
    for ax, doc in zip(axes, docs):
        group = rank1.query('document == @doc')
        ax.boxplot(group['length_ratio'].dropna(), vert=True, labels=[doc])
        ax.axhline(1.0, color='grey', linestyle='--', linewidth=1)
        ax.set_ylabel('Auto / Manual char length ratio')
        ax.set_title(f'{doc} ({scenario})')
    plt.tight_layout()
    plt.show()


## Auto Paragraph Alignment Overview

In [ ]:

coverage_tables = []
for scenario, result in analysis_results.items():
    coverage = result['coverage']
    if coverage.empty:
        continue
    coverage = coverage.copy()
    coverage['analysis'] = scenario
    coverage_tables.append(coverage)

auto_coverage = pd.concat(coverage_tables, ignore_index=True) if coverage_tables else pd.DataFrame()
auto_coverage


In [ ]:

for scenario, result in analysis_results.items():
    auto_best = result['auto_best']
    if auto_best.empty:
        print(f"{scenario}: No auto paragraphs matched")
        continue
    docs = auto_best['document'].unique()
    fig, axes = plt.subplots(len(docs), 1, figsize=(12, 5 * len(docs)), sharex=False)
    if len(docs) == 1:
        axes = [axes]
    for ax, doc in zip(axes, docs):
        group = auto_best.query('document == @doc').sort_values('auto_para_index')
        ax.plot(group['auto_para_index'], group['cosine_similarity'], marker='o', linestyle='-', label=doc)
        ax.set_title(f'Best Manual Match per DeepSeek Paragraph ({doc}, {scenario})')
        ax.set_xlabel('DeepSeek para_index')
        ax.set_ylabel('Cosine similarity')
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:

for scenario, result in analysis_results.items():
    auto_best = result['auto_best']
    if auto_best.empty:
        continue
    docs = auto_best['document'].unique()
    fig, axes = plt.subplots(len(docs), 1, figsize=(12, 5 * len(docs)), sharex=False)
    if len(docs) == 1:
        axes = [axes]
    for ax, doc in zip(axes, docs):
        group = auto_best.query('document == @doc').sort_values('auto_para_index')
        ax.bar(group['auto_para_index'], group['length_ratio'], color='#4c72b0')
        ax.axhline(1.0, color='grey', linestyle='--', linewidth=1)
        ax.set_title(f'Length Ratio by DeepSeek para_index ({doc}, {scenario})')
        ax.set_xlabel('DeepSeek para_index')
        ax.set_ylabel('Auto / Manual char length')
    plt.tight_layout()
    plt.show()


## Lowest Similarity Matches (rank 1)

In [ ]:

low_records = []
for scenario, result in analysis_results.items():
    rank1 = result['rank1']
    if rank1.empty:
        continue
    low = rank1.sort_values('cosine_similarity').head(10).copy()
    low['analysis'] = scenario
    low_records.append(low[["analysis", "document", "manual_id", "auto_id", "cosine_similarity", "sequence_ratio", "jaccard_similarity", "manual_preview", "auto_preview"]])

low_matches = pd.concat(low_records, ignore_index=True) if low_records else pd.DataFrame()
low_matches


## Highest Similarity Matches (rank 1)

In [ ]:

high_records = []
for scenario, result in analysis_results.items():
    rank1 = result['rank1']
    if rank1.empty:
        continue
    high = rank1.sort_values('cosine_similarity', ascending=False).head(10).copy()
    high['analysis'] = scenario
    high_records.append(high[["analysis", "document", "manual_id", "auto_id", "cosine_similarity", "sequence_ratio", "jaccard_similarity", "manual_preview", "auto_preview"]])

high_matches = pd.concat(high_records, ignore_index=True) if high_records else pd.DataFrame()
high_matches
